## Step 1 — Debug: print shapes at every step inside forward()

In [ ]:
# Run this cell to see EXACTLY where the shape breaks.
# We add print() inside a debug version of forward().

import torch, torch.nn as nn, torch.nn.functional as F
from torch_geometric.nn import CGConv, global_mean_pool

class CharlesCGCNN_DEBUG(nn.Module):
    """
    Identical to CharlesCGCNN but with print() at every step.
    Run ONE batch through this to see where the shape goes wrong.
    """
    def __init__(self, node_feat_dim=2, edge_feat_dim=1,
                 hidden_dim=64, global_dim=11, num_classes=2):
        super().__init__()
        # ── THE KEY CHANGE ──────────────────────────────────────────────────
        # CGConv with an INTEGER for channels keeps in_dim == out_dim.
        # To go from 2-dim atom features to 64-dim we need ONE Linear first.
        # This is exactly what the original CGCNN paper (Xie & Grossman 2018)
        # calls the "atom feature vector initialisation layer".
        self.node_emb = nn.Linear(node_feat_dim, hidden_dim)  # 2 → 64
        # Now both conv layers use integer channels — in = out = hidden_dim
        self.conv1 = CGConv(hidden_dim, dim=edge_feat_dim)     # 64 → 64
        self.conv2 = CGConv(hidden_dim, dim=edge_feat_dim)     # 64 → 64
        self.fc    = nn.Linear(hidden_dim + global_dim, num_classes)  # 75 → 2

    def forward(self, data):
        x, ei, ea, batch, u = (data.x, data.edge_index,
                                data.edge_attr, data.batch, data.u)
        if u.dim() == 3: u = u.squeeze(1)   # [B,1,11] → [B,11]

        print(f"  [0] x after load       : {x.shape}")   # [N, 2]
        x = F.relu(self.node_emb(x))
        print(f"  [1] x after node_emb   : {x.shape}")   # [N, 64]
        x = F.relu(self.conv1(x, ei, ea))
        print(f"  [2] x after conv1      : {x.shape}")   # [N, 64]
        x = F.relu(self.conv2(x, ei, ea))
        print(f"  [3] x after conv2      : {x.shape}")   # [N, 64]
        x = global_mean_pool(x, batch)
        print(f"  [4] x after pool       : {x.shape}")   # [B, 64]
        print(f"  [5] u                  : {u.shape}")    # [B, 11]
        out = torch.cat([x, u], dim=1)
        print(f"  [6] out after cat      : {out.shape}")  # [B, 75]
        out = self.fc(out)
        print(f"  [7] out after fc       : {out.shape}")  # [B, 2]
        return out

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dbg = CharlesCGCNN_DEBUG().to(device)
dbg.eval()

sample = next(iter(train_loader)).to(device)
print("=== Debug forward pass ===")
with torch.no_grad():
    out = dbg(sample)
print(f"\nFinal output shape: {out.shape}  ← should be [{sample.num_graphs}, 2]")

## Step 2 — CharlesCGCNN (fixed, clean version)

The only change from the broken version:
- Added `self.node_emb = nn.Linear(2, 64)` — projects atom features to 64 dims
- Changed `CGConv(channels=(2,64), dim=1)` → `CGConv(64, dim=1)` — integer, not tuple
- `CGConv` with a tuple `(in, out)` is for **bipartite** graphs; integer is for homogeneous graphs
- This is exactly how Xie & Grossman (2018) built the original CGCNN

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch_geometric.nn import CGConv, global_mean_pool
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
import os

class CharlesCGCNN(nn.Module):
    def __init__(self, node_feat_dim=2, edge_feat_dim=1,
                 hidden_dim=64, global_dim=11, num_classes=2):
        super().__init__()
        # Step 1: embed raw atom features [Z, EN] into hidden_dim space
        self.node_emb = nn.Linear(node_feat_dim, hidden_dim)
        # Step 2: two CGConv message-passing layers at hidden_dim
        self.conv1 = CGConv(hidden_dim, dim=edge_feat_dim)
        self.conv2 = CGConv(hidden_dim, dim=edge_feat_dim)
        # Step 3: classify from (pooled_graph_embed + global_features)
        self.fc = nn.Linear(hidden_dim + global_dim, num_classes)

    def forward(self, data):
        x, ei, ea, batch, u = (data.x, data.edge_index,
                                data.edge_attr, data.batch, data.u)
        if u.dim() == 3: u = u.squeeze(1)   # guard for [B,1,11] edge case
        # atom embedding
        x = F.relu(self.node_emb(x))         # [N_atoms, 64]
        # message passing — each atom learns from its bonded neighbours
        x = F.relu(self.conv1(x, ei, ea))    # [N_atoms, 64]
        x = F.relu(self.conv2(x, ei, ea))    # [N_atoms, 64]
        # pool all atoms in each crystal into one vector
        x = global_mean_pool(x, batch)       # [B, 64]
        # append global (lattice / electronic) descriptors
        out = torch.cat([x, u], dim=1)       # [B, 75]
        return self.fc(out)                  # [B, 2]

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
charles_cgcnn = CharlesCGCNN(
    node_feat_dim=2, edge_feat_dim=1,
    hidden_dim=64, global_dim=N_GLOBAL, num_classes=2
).to(device)

# ── Quick shape sanity check ───────────────────────────────────────────────────
charles_cgcnn.eval()
with torch.no_grad():
    s = next(iter(train_loader)).to(device)
    out = charles_cgcnn(s)
print(f"Forward pass OK: input atoms {s.x.shape} → output {out.shape}")
# Expect: output torch.Size([32, 2])
n = sum(p.numel() for p in charles_cgcnn.parameters() if p.requires_grad)
print(f"Total parameters: {n:,}")

## Step 3 — Train CharlesCGCNN

In [ ]:
cgcnn_optimizer = torch.optim.Adam(
    charles_cgcnn.parameters(), lr=1e-3, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()
cgcnn_history = {'train_loss':[], 'val_acc':[], 'val_f1':[], 'val_auc':[]}
NUM_EPOCHS = 50

print(f"Training CharlesCGCNN for {NUM_EPOCHS} epochs on {device}...")
print("-"*65)

for epoch in range(1, NUM_EPOCHS+1):

    # ── train ──────────────────────────────────────────────────────────────
    charles_cgcnn.train()
    total_loss = 0.0
    for batch in train_loader:
        batch = batch.to(device)
        cgcnn_optimizer.zero_grad()
        out  = charles_cgcnn(batch)
        loss = criterion(out, batch.y)
        loss.backward()
        cgcnn_optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    avg_loss = total_loss / len(train_dataset)

    # ── validate ────────────────────────────────────────────────────────────
    charles_cgcnn.eval()
    preds, probs, labels = [], [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            out   = charles_cgcnn(batch)
            preds.extend(out.argmax(dim=1).cpu().numpy())
            probs.extend(torch.softmax(out,dim=1)[:,1].cpu().numpy())
            labels.extend(batch.y.cpu().numpy())

    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, zero_division=0)
    auc = roc_auc_score(labels, probs)
    cgcnn_history['train_loss'].append(avg_loss)
    cgcnn_history['val_acc'].append(acc)
    cgcnn_history['val_f1'].append(f1)
    cgcnn_history['val_auc'].append(auc)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{NUM_EPOCHS} | Loss: {avg_loss:.4f} | "
              f"Val Acc: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")

# ── test ────────────────────────────────────────────────────────────────────
charles_cgcnn.eval()
tp, tprob, tlab = [], [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        out   = charles_cgcnn(batch)
        tp.extend(out.argmax(dim=1).cpu().numpy())
        tprob.extend(torch.softmax(out,dim=1)[:,1].cpu().numpy())
        tlab.extend(batch.y.cpu().numpy())

print("\n=== CharlesCGCNN — TEST SET ===")
print(f"  Accuracy : {accuracy_score(tlab, tp):.4f}")
print(f"  F1-score : {f1_score(tlab, tp):.4f}")
print(f"  ROC-AUC  : {roc_auc_score(tlab, tprob):.4f}")
print(classification_report(tlab, tp, target_names=['Trivial','Topological']))

# save checkpoint to Drive
save_path = os.path.join(BASE_DIR, 'Charles_CGCNN.pth')
torch.save(charles_cgcnn.state_dict(), save_path)
print(f"Saved → {save_path}")

## Step 4 — CharlesAttentionGNN (same fix applied)

`GATConv` also requires a consistent `in_channels`. Same pattern: `node_emb` first, then GAT layers at `hidden_dim`.

In [ ]:
from torch_geometric.nn import GATConv, global_mean_pool

class CharlesAttentionGNN(nn.Module):
    def __init__(self, node_feat_dim=2, edge_feat_dim=1,
                 hidden_dim=64, heads=4, global_dim=11, num_classes=2):
        super().__init__()
        self.node_emb = nn.Linear(node_feat_dim, hidden_dim)   # 2 → 64
        # GAT layer 1: 64 → 64*heads (multi-head, concatenated)
        self.gat1 = GATConv(hidden_dim, hidden_dim,
                             heads=heads, edge_dim=edge_feat_dim, concat=True)
        # GAT layer 2: 64*heads → 64 (single head, averaged)
        self.gat2 = GATConv(hidden_dim*heads, hidden_dim,
                             heads=1, concat=False)
        self.fc = nn.Linear(hidden_dim + global_dim, num_classes)

    def forward(self, data, return_attention=False):
        x, ei, ea, batch, u = (data.x, data.edge_index,
                                data.edge_attr, data.batch, data.u)
        if u.dim() == 3: u = u.squeeze(1)
        x = F.relu(self.node_emb(x))                  # [N, 64]
        if return_attention:
            x, (att_ei, alpha) = self.gat1(
                x, ei, ea, return_attention_weights=True)
        else:
            x = self.gat1(x, ei, ea)
        x = F.elu(x)                                   # [N, 64*heads]
        x = F.elu(self.gat2(x, ei))                    # [N, 64]
        x = global_mean_pool(x, batch)                 # [B, 64]
        out = torch.cat([x, u], dim=1)                 # [B, 75]
        out = self.fc(out)
        if return_attention:
            return out, att_ei, alpha
        return out

torch.manual_seed(42)
charles_attn = CharlesAttentionGNN(
    node_feat_dim=2, edge_feat_dim=1, hidden_dim=64,
    heads=4, global_dim=N_GLOBAL, num_classes=2).to(device)

charles_attn.eval()
with torch.no_grad():
    s = next(iter(train_loader)).to(device)
    o = charles_attn(s)
print(f"AttentionGNN forward OK: {s.x.shape} → {o.shape}")

In [ ]:
attn_optimizer = torch.optim.Adam(
    charles_attn.parameters(), lr=1e-3, weight_decay=1e-5)
attn_history = {'train_loss':[], 'val_acc':[], 'val_f1':[], 'val_auc':[]}

NUM_EPOCHS_ATTN = 50
print(f"Training CharlesAttentionGNN for {NUM_EPOCHS_ATTN} epochs...")
print("-"*65)

for epoch in range(1, NUM_EPOCHS_ATTN+1):
    charles_attn.train()
    total_loss = 0.0
    for batch in train_loader:
        batch = batch.to(device)
        attn_optimizer.zero_grad()
        out  = charles_attn(batch)
        loss = criterion(out, batch.y)
        loss.backward(); attn_optimizer.step()
        total_loss += loss.item() * batch.num_graphs

    charles_attn.eval()
    preds, probs, labels = [], [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device); out = charles_attn(batch)
            preds.extend(out.argmax(dim=1).cpu().numpy())
            probs.extend(torch.softmax(out,dim=1)[:,1].cpu().numpy())
            labels.extend(batch.y.cpu().numpy())
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, zero_division=0)
    auc = roc_auc_score(labels, probs)
    attn_history['train_loss'].append(total_loss/len(train_dataset))
    attn_history['val_acc'].append(acc); attn_history['val_f1'].append(f1)
    attn_history['val_auc'].append(auc)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{NUM_EPOCHS_ATTN} | "
              f"Loss: {total_loss/len(train_dataset):.4f} | "
              f"Val Acc: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")

charles_attn.eval()
tp, tprob, tlab = [], [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device); out = charles_attn(batch)
        tp.extend(out.argmax(dim=1).cpu().numpy())
        tprob.extend(torch.softmax(out,dim=1)[:,1].cpu().numpy())
        tlab.extend(batch.y.cpu().numpy())
print(f"\n=== AttentionGNN TEST | Acc: {accuracy_score(tlab,tp):.4f} "
      f"| F1: {f1_score(tlab,tp):.4f} | AUC: {roc_auc_score(tlab,tprob):.4f}")
torch.save(charles_attn.state_dict(), os.path.join(BASE_DIR,'Charles_AttentionGNN.pth'))
print("Saved AttentionGNN.")

## Step 5 — CharlesMEGNet (same fix applied)

In [ ]:
from torch_geometric.utils import scatter
from torch_geometric.nn import Set2Set

class MEGNetBlock(nn.Module):
    def __init__(self, node_dim, edge_dim, global_dim, hidden):
        super().__init__()
        self.edge_mlp = nn.Sequential(
            nn.Linear(edge_dim + 2*node_dim + global_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden))
        self.node_mlp = nn.Sequential(
            nn.Linear(node_dim + hidden + global_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden))
        self.global_mlp = nn.Sequential(
            nn.Linear(global_dim + hidden + hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden))

    def forward(self, x, ei, ea, u, batch):
        src, dst = ei
        u_e = u[batch[src]]
        ea_new = self.edge_mlp(torch.cat([ea, x[src], x[dst], u_e], dim=1))
        agg    = scatter(ea_new, dst, dim=0, dim_size=x.size(0), reduce='mean')
        x_new  = self.node_mlp(torch.cat([x, agg, u[batch]], dim=1))
        mn     = scatter(x_new, batch, dim=0, dim_size=u.size(0), reduce='mean')
        me     = scatter(ea_new, batch[src], dim=0, dim_size=u.size(0), reduce='mean')
        u_new  = self.global_mlp(torch.cat([u, mn, me], dim=1))
        return x_new, ea_new, u_new

class CharlesMEGNet(nn.Module):
    def __init__(self, node_feat_dim=2, edge_feat_dim=1,
                 global_dim=11, hidden_dim=64, n_blocks=3, num_classes=2):
        super().__init__()
        # embed raw node and edge features to hidden_dim BEFORE the blocks
        self.node_emb = nn.Linear(node_feat_dim, hidden_dim)   # 2 → 64
        self.edge_emb = nn.Linear(edge_feat_dim, hidden_dim)   # 1 → 64
        self.blocks = nn.ModuleList([
            MEGNetBlock(hidden_dim, hidden_dim,
                        global_dim if i==0 else hidden_dim, hidden_dim)
            for i in range(n_blocks)])
        self.pool = Set2Set(hidden_dim, processing_steps=3)     # → 2*hidden
        self.fc   = nn.Linear(2*hidden_dim + hidden_dim, num_classes)  # 192 → 2

    def forward(self, data):
        x, ei, ea, u, batch = (data.x, data.edge_index, data.edge_attr,
                                data.u, data.batch)
        if u.dim() == 3: u = u.squeeze(1)
        x  = F.relu(self.node_emb(x))    # [N, 64]
        ea = F.relu(self.edge_emb(ea))   # [E, 64]
        for block in self.blocks:
            x, ea, u = block(x, ei, ea, u, batch)
        xp  = self.pool(x, batch)        # [B, 128]
        out = torch.cat([xp, u], dim=1)  # [B, 192]
        return self.fc(out)              # [B, 2]

torch.manual_seed(42)
charles_megnet = CharlesMEGNet(
    node_feat_dim=2, edge_feat_dim=1, global_dim=N_GLOBAL,
    hidden_dim=64, n_blocks=3, num_classes=2).to(device)

charles_megnet.eval()
with torch.no_grad():
    s = next(iter(train_loader)).to(device)
    o = charles_megnet(s)
print(f"MEGNet forward OK: {s.x.shape} → {o.shape}")
print(f"Parameters: {sum(p.numel() for p in charles_megnet.parameters() if p.requires_grad):,}")

In [ ]:
meg_optimizer = torch.optim.Adam(
    charles_megnet.parameters(), lr=1e-3, weight_decay=1e-5)
meg_history = {'train_loss':[], 'val_acc':[], 'val_f1':[], 'val_auc':[]}

NUM_EPOCHS_MEG = 50
print(f"Training CharlesMEGNet for {NUM_EPOCHS_MEG} epochs...")
print("-"*65)

for epoch in range(1, NUM_EPOCHS_MEG+1):
    charles_megnet.train()
    total_loss = 0.0
    for batch in train_loader:
        batch = batch.to(device)
        meg_optimizer.zero_grad()
        out  = charles_megnet(batch)
        loss = criterion(out, batch.y)
        loss.backward(); meg_optimizer.step()
        total_loss += loss.item() * batch.num_graphs

    charles_megnet.eval()
    preds, probs, labels = [], [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device); out = charles_megnet(batch)
            preds.extend(out.argmax(dim=1).cpu().numpy())
            probs.extend(torch.softmax(out,dim=1)[:,1].cpu().numpy())
            labels.extend(batch.y.cpu().numpy())
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, zero_division=0)
    auc = roc_auc_score(labels, probs)
    meg_history['train_loss'].append(total_loss/len(train_dataset))
    meg_history['val_acc'].append(acc); meg_history['val_f1'].append(f1)
    meg_history['val_auc'].append(auc)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{NUM_EPOCHS_MEG} | "
              f"Loss: {total_loss/len(train_dataset):.4f} | "
              f"Val Acc: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")

charles_megnet.eval()
tp, tprob, tlab = [], [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device); out = charles_megnet(batch)
        tp.extend(out.argmax(dim=1).cpu().numpy())
        tprob.extend(torch.softmax(out,dim=1)[:,1].cpu().numpy())
        tlab.extend(batch.y.cpu().numpy())
print(f"\n=== MEGNet TEST | Acc: {accuracy_score(tlab,tp):.4f} "
      f"| F1: {f1_score(tlab,tp):.4f} | AUC: {roc_auc_score(tlab,tprob):.4f}")
print(classification_report(tlab,tp,target_names=['Trivial','Topological']))
torch.save(charles_megnet.state_dict(), os.path.join(BASE_DIR,'Charles_MEGNet.pth'))
print("Saved MEGNet.")